In [1]:
import pandas as pd

# Leer el archivo segmentos.csv
df_segmentos = pd.read_csv('data/segmentos.csv')


df_agrupado = df_segmentos.groupby('subida')


print("\nResumen por subida:")
resumen = df_agrupado.agg({
    'denivel': 'first',
    'pendiente': 'first',
    'rider': 'count'  # Cantidad de ciclistas por subida
}).rename(columns={'rider': 'cantidad_riders'})
print(resumen)
# Convertir minutos a float y calcular el tiempo total en minutos
df_segmentos['minutos'] = pd.to_numeric(df_segmentos['minutos'], errors='coerce')
df_segmentos['tiempo_total_min'] = df_segmentos['minutos'] + (df_segmentos['segundos'] / 60)
# Calcular VAM (m/h)
df_segmentos['VAM_m_h'] = ((df_segmentos['denivel'] / df_segmentos['tiempo_total_min']) * 60).round(2)
# Calcular factor_grado y w_kg y agregarlos al dataframe
df_segmentos['factor_grado'] = 2 + (df_segmentos['pendiente'] / 10)

df_segmentos['w_kg'] = (df_segmentos['VAM_m_h'] / (df_segmentos['factor_grado'] * 100)).round(2)
# Mostrar estadísticas por subida
print("\nEstadísticas de W/kg por subida:")
stats_subida = df_segmentos.groupby('subida')['w_kg'].agg(['count', 'mean', 'min', 'max', 'std'])
stats_subida = stats_subida.round(2)
print(stats_subida)
df_sorted = df_segmentos.sort_values(['subida', 'w_kg'], ascending=[True, False])
subidas_unicas = df_segmentos['subida'].unique()


print("\n" + "="*80)
print("DETALLE DE W/KG POR SUBIDA Y CICLISTA")
print("="*80)
for subida in df_segmentos['subida'].unique():
    df_subida = df_segmentos[df_segmentos['subida'] == subida][['rider', 'minutos','segundos','w_kg', 'VAM_m_h', 'denivel', 'pendiente']].sort_values('w_kg', ascending=False)
    print(f"\n{subida.upper()}:")
    print(df_subida.to_string(index=False))
    print(f"  Promedio W/kg: {df_subida['w_kg'].mean():.2f} | Mejor: {df_subida['w_kg'].max():.2f} | Peor: {df_subida['w_kg'].min():.2f}")
# Obtener lista única de ciclistas
ciclistas = df_segmentos['rider'].unique()
subidas_list = sorted(df_segmentos['subida'].unique())

# Crear diccionario de posiciones para subidas (para el eje X)
subida_pos = {subida: i for i, subida in enumerate(subidas_list)}


print("\nResumen comparativo de ciclistas:")
print("="*80)
resumen_ciclistas = df_segmentos.groupby('rider')['w_kg'].agg(['mean', 'max', 'min', 'count']).round(1)
resumen_ciclistas = resumen_ciclistas.sort_values('mean', ascending=False)
resumen_ciclistas.columns = ['W/kg Promedio', 'W/kg Máximo', 'W/kg Mínimo', 'Subidas']
print(resumen_ciclistas)


Resumen por subida:
                            denivel  pendiente  cantidad_riders
subida                                                         
Juanar                          287        5.7                8
LA CONCEPCION                   293        7.6                9
La Frontera hasta el Garbí      447        8.9               14
Las millanas-jorox              413        5.0               17
Repetidor                       417       10.2               13
Subida Bobastro Presa Alta      338        7.9               15

Estadísticas de W/kg por subida:
                            count  mean   min   max   std
subida                                                   
Juanar                          8  5.43  4.50  5.78  0.47
LA CONCEPCION                   9  5.60  4.48  5.86  0.45
La Frontera hasta el Garbí     14  5.56  5.27  5.78  0.18
Las millanas-jorox             17  6.37  5.49  6.88  0.39
Repetidor                      13  4.94  4.15  5.47  0.41
Subida Bobastro Presa Alta 